# Experiment: Plate Raster Synchrony Viewer

Objective:
- Discover analyzed scan outputs under the `.../Network/<scan_id>/wellXYZ/` pattern.
- Render one interactive 4x6 plate view at a time with raster, synchrony, or both in each well.
- Switch the current plate view with a single `run_id` dropdown instead of creating one preview cell per scan.
- Use a shared top x-span selector to sweep the visible time window across the whole 24-well plate.
- Export publication-style PNGs plus portable offline HTML for review.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

import plotly.io as pio

# Add workspace root to path for robust imports in notebook environment
_workspace_root = Path('/mnt/Vol20tb1/yuxin/codes/Yuxin_Fork')
if str(_workspace_root) not in sys.path:
    sys.path.insert(0, str(_workspace_root))

from IPNAnalysis.workbooks.Yuxin.plate_raster_synchrony_viewer_utils import (
    DISPLAY_MODES,
    build_run_manifest,
    choose_notebook_renderer,
    compute_plate_height_px,
    create_run_figure,
    create_scan_figure,
    discover_well_records,
    export_all_scans,
    export_scan_figure,
    render_run_viewer,
    summarize_scans,
)


## Notes

- Expected analyzed layout: `<root>/.../Network/<scan_id>/wellXYZ/`
- Plot inputs per well: `spike_times.npy` and `network_results.json`
- Plate mapping is fixed to row-major order: `well000 -> A1`, `well001 -> A2`, ..., `well023 -> D6`
- Each well overlays raster spikes with network synchrony in the same panel; synchrony is drawn on top.
- PNG export uses Plotly image export and typically requires `kaleido`.


In [2]:
# Plotting and export helpers live in `plate_raster_synchrony_viewer_utils.py`.
# This keeps the notebook focused on configuration, preview, and export.
DISPLAY_MODES


{'both', 'raster', 'synchrony'}

In [3]:
# Imported helpers:
# - discover_well_records / build_run_manifest / summarize_scans
# - create_run_figure / create_scan_figure / render_run_viewer
# - export_scan_figure / export_all_scans


In [4]:
# The figure builder now creates a subplot-backed 4x6 plate with matched x-axes,
# fixed per-subplot y-axes, a top x-span selector, and a lazy run_id dropdown viewer.


In [5]:
# Export helpers now emit offline HTML and export-safe PNGs.


## Configure paths and rendering

Set `ANALYSIS_ROOT` to the analyzed data directory that contains `.../Network/<scan_id>/wellXYZ/` outputs.
The unified preview viewer requires `ipywidgets` in the notebook environment.
Preview and export now use fixed canvas dimensions sized for a 2:1 width:height ratio in each well subplot.


In [ ]:
ANALYSIS_ROOT = Path('/mnt/benshalom-nas/analysis/Sadegh/CX138')
OUTPUT_DIR = Path('./plate_exports')

DISPLAY_MODE = 'both'          # 'raster', 'synchrony', or 'both'
EXPORT_DPI = 600               # user-configurable static export DPI
FIGURE_WIDTH_IN = 24.0         # export width in inches
FIGURE_HEIGHT_IN = compute_plate_height_px(int(FIGURE_WIDTH_IN * 100)) / 100.0
PREVIEW_WIDTH_PX = 2400        # fixed interactive notebook display width
PREVIEW_HEIGHT_PX = compute_plate_height_px(PREVIEW_WIDTH_PX)
INITIAL_WINDOW_S = 300.0        # initial shared x-window / top selector value
MARKER_SIZE = 5.0
LINE_WIDTH = 1.25
UNIT_SORT_MODE = 'firing_rate_desc'
PREVIEW_RENDERER = 'plotly_mimetype'
PREVIEW_MAX_RASTER_POINTS_PER_WELL = 12000
PREVIEW_MAX_SYNCHRONY_POINTS = 3000

EXPORT_HTML = True
EXPORT_PNG = True


In [7]:
index_df = discover_well_records(ANALYSIS_ROOT)
run_manifest_df = build_run_manifest(index_df)
summary_df = summarize_scans(index_df)
print(f'Found {len(run_manifest_df)} run(s) and {len(index_df)} well record(s).')
display(run_manifest_df)


Found 19 run(s) and 456 well record(s).


,run_id,scan_label,scan_dir,n_wells,missing_spike_times,missing_network_json
0,000004,CX138/260319/T003346/Network/000004,/mnt/benshalom-nas/analysis/Sadegh/CX138/CX138...,24,0,0
1,000005,CX138/260319/T003346/Network/000005,/mnt/benshalom-nas/analysis/Sadegh/CX138/CX138...,24,0,0
2,000007,CX138/260321/T003346/Network/000007,/mnt/benshalom-nas/analysis/Sadegh/CX138/CX138...,24,0,0
3,000009,CX138/260323/T003346/Network/000009,/mnt/benshalom-nas/analysis/Sadegh/CX138/CX138...,24,0,0
4,000012,CX138/260325/T003346/Network/000012,/mnt/benshalom-nas/analysis/Sadegh/CX138/CX138...,24,0,0
5,000014,CX138/260325/T003346/Network/000014,/mnt/benshalom-nas/analysis/Sadegh/CX138/CX138...,24,0,0
6,000016,CX138/260326/T003346/Network/000016,/mnt/benshalom-nas/analysis/Sadegh/CX138/CX138...,24,0,0
7,000019,CX138/260327/T003346/Network/000019,/mnt/benshalom-nas/analysis/Sadegh/CX138/CX138...,24,0,0
8,000027,CX138/260328/T003346/Network/000027,/mnt/benshalom-nas/analysis/Sadegh/CX138/CX138...,24,0,0
9,000029,CX138/260329/T003346/Network/000029,/mnt/benshalom-nas/analysis/Sadegh/CX138/CX138...,24,0,0


## Preview all runs interactively

Use the `run_id` dropdown to switch the currently displayed run.
Use the top integer selector inside the figure to adjust the shared x-axis window across all 24 wells at once.


In [8]:
INITIAL_RUN_ID = None

if summary_df.empty:
    raise RuntimeError('No runs found. Check ANALYSIS_ROOT and the expected directory pattern.')

viewer = render_run_viewer(
    index_df,
    display_mode=DISPLAY_MODE,
    marker_size=MARKER_SIZE,
    line_width=LINE_WIDTH,
    width_px=PREVIEW_WIDTH_PX,
    height_px=PREVIEW_HEIGHT_PX,
    unit_sort_mode=UNIT_SORT_MODE,
    max_raster_points_per_well=PREVIEW_MAX_RASTER_POINTS_PER_WELL,
    max_synchrony_points=PREVIEW_MAX_SYNCHRONY_POINTS,
    initial_run_id=INITIAL_RUN_ID,
    initial_window_s=INITIAL_WINDOW_S,
    preferred_renderer=PREVIEW_RENDERER,
)
viewer


## Export current scan or all scans

- `export_scan_figure(...)` saves the current interactive figure as offline HTML and/or PNG.
- `export_all_scans(...)` renders and exports every discovered scan using the current configuration.


In [9]:
RUN_ID_TO_EXPORT = summary_df.iloc[0]['run_id'] if not summary_df.empty else None

if RUN_ID_TO_EXPORT is None:
    raise RuntimeError('No runs available to export.')

EXPORT_PREVIEW_WIDTH_PX = max(1200, int(FIGURE_WIDTH_IN * 100))
EXPORT_PREVIEW_HEIGHT_PX = compute_plate_height_px(EXPORT_PREVIEW_WIDTH_PX)

export_fig = create_run_figure(
    index_df,
    RUN_ID_TO_EXPORT,
    display_mode=DISPLAY_MODE,
    marker_size=MARKER_SIZE,
    line_width=LINE_WIDTH,
    width_px=EXPORT_PREVIEW_WIDTH_PX,
    height_px=EXPORT_PREVIEW_HEIGHT_PX,
    unit_sort_mode=UNIT_SORT_MODE,
    max_raster_points_per_well=PREVIEW_MAX_RASTER_POINTS_PER_WELL,
    max_synchrony_points=PREVIEW_MAX_SYNCHRONY_POINTS,
    initial_window_s=INITIAL_WINDOW_S,
)
CURRENT_STEM = f'{RUN_ID_TO_EXPORT}_plate_overlay'

# Export the currently configured run_id.
# exported_paths = export_scan_figure(
#     export_fig,
#     output_dir=OUTPUT_DIR,
#     stem=CURRENT_STEM,
#     export_png=EXPORT_PNG,
#     export_html=EXPORT_HTML,
#     dpi=EXPORT_DPI,
#     width_in=FIGURE_WIDTH_IN,
#     height_in=FIGURE_HEIGHT_IN,
# )
# exported_paths

# Export every discovered scan.
# all_exports = export_all_scans(
#     index_df,
#     output_dir=OUTPUT_DIR,
#     display_mode=DISPLAY_MODE,
#     dpi=EXPORT_DPI,
#     width_in=FIGURE_WIDTH_IN,
#     height_in=FIGURE_HEIGHT_IN,
#     marker_size=MARKER_SIZE,
#     line_width=LINE_WIDTH,
#     unit_sort_mode=UNIT_SORT_MODE,
#     max_raster_points_per_well=PREVIEW_MAX_RASTER_POINTS_PER_WELL,
#     max_synchrony_points=PREVIEW_MAX_SYNCHRONY_POINTS,
#     export_html=EXPORT_HTML,
#     export_png=EXPORT_PNG,
#     initial_window_s=INITIAL_WINDOW_S,
# )
# pd.DataFrame(all_exports)


In [10]:
print("Available renderers:", set(pio.renderers))
print("Default renderer:", pio.renderers.default)


Available renderers: {'vscode', 'sphinx_gallery', 'cocalc', 'iframe', 'browser', 'databricks', 'svg', 'jupyterlab', 'notebook_connected', 'chrome', 'kaggle', 'jpeg', 'nteract', 'azure', 'sphinx_gallery_png', 'json', 'chromium', 'firefox', 'pdf', 'colab', 'notebook', 'jpg', 'iframe_connected', 'plotly_mimetype', 'png'}
Default renderer: plotly_mimetype
